# Day 9 v3 — Silver: Invoice Metadata (Production — Job Parameters Only)

**Production notebook — ADF sends `load_type` only; date params auto-compute.**

---

## What this notebook does

Scans Bronze Volume PDF file paths directly.
Bronze stores raw PDFs only — there is no `_metadata` Delta table.
Metadata is extracted from the file path pattern: `invoices/YYYY/MM/DD/INV-AU-YYYY-NNNN.pdf`

Silver stores: `invoice_id`, `invoice_date`, `invoice_number`, `invoice_year/month/day`, `file_size_kb`, `bronze_path`

PDF content (amounts, line items) is not parsed here — that belongs in Gold.

---

## Parameters (ADF `baseParameters`)

| Parameter | Required | Example | Notes |
|---|---|---|---|
| `load_type` | Yes | `incremental` | `full` or `incremental` |
| `ingestion_year` | No | `2026` | default: yesterday UTC year |
| `ingestion_month` | No | `06` | default: yesterday UTC month |
| `ingestion_day` | No | `01` | default: yesterday UTC day |

---

## Data flow

```
Bronze: bronze-volume/invoices/YYYY/MM/DD/INV-AU-YYYY-NNNN.pdf  (raw PDFs)

    full  → scan bronze-volume/invoices/ recursively
    incr  → scan bronze-volume/invoices/YYYY/MM/DD/ only

    extract invoice_id, year, month, day, file_size_kb from path+size
    build invoice_date, extract invoice_number
    data quality → quarantine
    deduplicate on invoice_id → Delta MERGE

Silver: silver-volume/invoices/             (Delta — MERGE on invoice_id)
Silver: silver-volume/quarantine/invoices/  (Delta — rejected rows)
```

In [ ]:
# Cell 1 — Imports
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime, timezone

print("Imports OK")

In [ ]:
# Cell 2 — Parameters
# Only load_type is required from ADF / Databricks Job.
# Date params default to the PREVIOUS UTC day — today's invoices are still being
# generated when the daily job triggers, so we always process yesterday's completed data.
#
#   load_type       : "full" | "incremental"  (pass from ADF, default: incremental)
#   ingestion_year  : YYYY                    (default: yesterday's UTC year)
#   ingestion_month : MM                      (default: yesterday's UTC month)
#   ingestion_day   : DD                      (default: yesterday's UTC day)

from datetime import timedelta
_now  = datetime.now(timezone.utc)
_prev = _now - timedelta(days=1)    # always process the completed prior day

def _get_param(key, default):
    try:
        val = dbutils.widgets.get(key).strip()
        return val if val else default
    except Exception:
        return default

LOAD_TYPE       = _get_param("load_type",       "incremental")
INGESTION_YEAR  = _get_param("ingestion_year",  str(_prev.year))
INGESTION_MONTH = _get_param("ingestion_month", f"{_prev.month:02d}")
INGESTION_DAY   = _get_param("ingestion_day",   f"{_prev.day:02d}")

if LOAD_TYPE not in ("full", "incremental"):
    raise ValueError(f"load_type must be 'full' or 'incremental', got: {LOAD_TYPE!r}")

print(f"load_type       : {LOAD_TYPE}")
print(f"ingestion_year  : {INGESTION_YEAR}")
print(f"ingestion_month : {INGESTION_MONTH}")
print(f"ingestion_day   : {INGESTION_DAY}")
print("Parameters resolved.")

In [ ]:
# Cell 3 — Constants
BRONZE_INVOICES = "/Volumes/dbw_ev_intelligence_dev/default/bronze-volume/invoices"
SILVER_PATH     = "/Volumes/dbw_ev_intelligence_dev/default/silver-volume/invoices"
QUARANTINE_PATH = "/Volumes/dbw_ev_intelligence_dev/default/silver-volume/quarantine/invoices"

PIPELINE_NAME = "job_silver_blob_invoices_v3"
RUN_TS        = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

print(f"Bronze invoices : {BRONZE_INVOICES}")
print(f"Silver path     : {SILVER_PATH}")
print(f"Quarantine path : {QUARANTINE_PATH}")
print(f"RUN_TS          : {RUN_TS}")

In [ ]:
# Cell 4 — Helper: recursively list PDF files under a Bronze path

def list_pdfs(path):
    """Return all FileInfo objects for .pdf files under `path` (recursive)."""
    results = []
    try:
        items = dbutils.fs.ls(path)
    except Exception:
        return results
    for item in items:
        if item.isDir():
            results.extend(list_pdfs(item.path))
        elif item.name.endswith(".pdf"):
            results.append(item)
    return results

def folder_exists_dbfs(path):
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False

print("Helpers defined.")

In [ ]:
# Cell 5 — Scan Bronze PDF files and build metadata rows
# Bronze layout: invoices/YYYY/MM/DD/INV-AU-YYYY-NNNNNNNN.pdf
# For incremental: scan only the specific YYYY/MM/DD folder.
# For full: scan the entire invoices/ folder recursively.

import re
from pyspark.sql import Row

if LOAD_TYPE == "incremental":
    scan_path = f"{BRONZE_INVOICES}/{INGESTION_YEAR}/{INGESTION_MONTH}/{INGESTION_DAY}"
    print(f"Mode: incremental — scanning {scan_path}")
else:
    scan_path = BRONZE_INVOICES
    print(f"Mode: full — scanning {scan_path}")

if not folder_exists_dbfs(scan_path):
    raise Exception(
        f"Bronze path not found: {scan_path}\n"
        f"Run day_6/04_bronze_blob_invoices_pdf.ipynb first for "
        f"{INGESTION_YEAR}/{INGESTION_MONTH}/{INGESTION_DAY}."
    )

pdf_files = list_pdfs(scan_path)
print(f"PDF files found: {len(pdf_files)}")

# Extract metadata from file path using regex
# Pattern: .../invoices/YYYY/MM/DD/INV-AU-YYYY-NNNNNNNN.pdf
rows = []
unmatched = []
for f in pdf_files:
    m = re.search(r"invoices/(\d{4})/(\d{2})/(\d{2})/(.+\.pdf)$", f.path)
    if m:
        rows.append(Row(
            invoice_id   = m.group(4).replace(".pdf", ""),
            year         = m.group(1),
            month        = m.group(2),
            day          = m.group(3),
            file_size_kb = round(f.size / 1024, 1),
            bronze_path  = f.path,
        ))
    else:
        unmatched.append(f.path)

if unmatched:
    print(f"WARNING: {len(unmatched)} files did not match path pattern — skipped:")
    for p in unmatched[:5]:
        print(f"  {p}")

BRONZE_ROW_COUNT = len(rows)
print(f"Metadata rows extracted: {BRONZE_ROW_COUNT}")

if BRONZE_ROW_COUNT == 0:
    raise Exception(
        f"No PDF files matched the expected path pattern under {scan_path}.\n"
        f"Expected: invoices/YYYY/MM/DD/INV-AU-YYYY-NNNNNNNN.pdf"
    )

raw_df = spark.createDataFrame(rows)
raw_df.show(3, truncate=False)

In [ ]:
# Cell 6 — Cast columns and build derived fields
# raw_df has: invoice_id(str), year(str), month(str), day(str), file_size_kb(float), bronze_path(str)
# Cast year/month/day to integer, build invoice_date as date, extract invoice_number.

cast_df = (
    raw_df
    .withColumn("invoice_id",    F.trim(F.col("invoice_id").cast("string")))
    .withColumn("invoice_year",  F.col("year").cast("integer"))
    .withColumn("invoice_month", F.col("month").cast("integer"))
    .withColumn("invoice_day",   F.col("day").cast("integer"))
    .withColumn("file_size_kb",  F.col("file_size_kb").cast("decimal(10,2)"))
    .withColumn("bronze_path",   F.trim(F.col("bronze_path").cast("string")))
    .drop("year", "month", "day")
    # invoice_date assembled from integer parts
    .withColumn(
        "invoice_date",
        F.to_date(
            F.concat_ws("-",
                F.col("invoice_year").cast("string"),
                F.lpad(F.col("invoice_month").cast("string"), 2, "0"),
                F.lpad(F.col("invoice_day").cast("string"),   2, "0"),
            ), "yyyy-MM-dd"
        )
    )
    # invoice_number: trailing digits of invoice_id (e.g. INV-AU-2026-0002266 -> 2266)
    .withColumn(
        "invoice_number",
        F.regexp_extract(F.col("invoice_id"), r"(\d+)$", 1).cast("long")
    )
)

print("After cast and enrichment:")
cast_df.printSchema()
cast_df.show(3, truncate=False)

In [ ]:
# Cell 7 — Data quality: quarantine helper

def write_quarantine(df, reject_reason):
    if df.limit(1).count() == 0:
        return
    (
        df.withColumn("reject_reason", F.lit(reject_reason))
          .withColumn("reject_ts",     F.lit(RUN_TS).cast("timestamp"))
          .write.format("delta").mode("append").option("mergeSchema", "true")
          .save(QUARANTINE_PATH)
    )

print("write_quarantine() defined")

In [ ]:
# Cell 8 — Data quality: quarantine rows with null invoice_id

null_id_df = cast_df.filter(
    F.col("invoice_id").isNull() | (F.col("invoice_id") == "")
)
null_id_count = null_id_df.count()
write_quarantine(null_id_df, "null_invoice_id")

after_id_df = cast_df.filter(
    F.col("invoice_id").isNotNull() & (F.col("invoice_id") != "")
)

print(f"Quarantined (null_invoice_id) : {null_id_count}")
print(f"Remaining                     : {after_id_df.count()}")

In [ ]:
# Cell 9 — Data quality: quarantine rows with null invoice_date
# A null invoice_date means year/month/day parts were corrupted.

null_date_df = after_id_df.filter(F.col("invoice_date").isNull())
null_date_count = null_date_df.count()
write_quarantine(null_date_df, "null_invoice_date")

clean_df = after_id_df.filter(F.col("invoice_date").isNotNull())

print(f"Quarantined (null_invoice_date) : {null_date_count}")
print(f"Clean rows remaining            : {clean_df.count()}")

In [ ]:
# Cell 10 — Data quality: quarantine rows with negative file_size_kb
# A negative file size means the metadata was corrupted at Bronze.

neg_size_df = clean_df.filter(
    F.col("file_size_kb").isNotNull() & (F.col("file_size_kb") < 0)
)
neg_size_count = neg_size_df.count()
write_quarantine(neg_size_df, "negative_file_size_kb")

clean_df = clean_df.filter(
    F.col("file_size_kb").isNull() | (F.col("file_size_kb") >= 0)
)

print(f"Quarantined (negative_file_size_kb) : {neg_size_count}")
print(f"Clean rows remaining                : {clean_df.count()}")

In [ ]:
# Cell 11 — Add Silver audit columns

audited_df = (
    clean_df
    .withColumn("silver_ingested_at", F.lit(RUN_TS).cast("timestamp"))
    .withColumn("silver_load_type",   F.lit(LOAD_TYPE))
    .withColumn("silver_pipeline",    F.lit(PIPELINE_NAME))
)

print(f"Rows with audit columns: {audited_df.count()}")

In [ ]:
# Cell 12 — Deduplicate on invoice_id (latest silver_ingested_at wins)

window = Window.partitionBy("invoice_id").orderBy(F.col("silver_ingested_at").desc())
deduped_df = (
    audited_df
    .withColumn("_row_num", F.row_number().over(window))
    .filter(F.col("_row_num") == 1)
    .drop("_row_num")
)

SILVER_ROW_COUNT = deduped_df.count()
print(f"After dedup: {SILVER_ROW_COUNT} rows")

In [ ]:
# Cell 13 — Write to Silver Delta (full overwrite or MERGE)

if LOAD_TYPE == "full" or not folder_exists_dbfs(SILVER_PATH):
    (
        deduped_df.write.format("delta")
        .mode("overwrite").option("overwriteSchema", "true")
        .save(SILVER_PATH)
    )
    print(f"Full overwrite complete — {SILVER_ROW_COUNT} rows written")
else:
    delta_table = DeltaTable.forPath(spark, SILVER_PATH)
    (
        delta_table.alias("target")
        .merge(
            deduped_df.alias("source"),
            "target.invoice_id = source.invoice_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print(f"MERGE complete — {SILVER_ROW_COUNT} rows upserted")

In [ ]:
# Cell 14 — Run summary

silver_df = spark.read.format("delta").load(SILVER_PATH)
silver_total = silver_df.count()

print("=" * 65)
print("SILVER INVOICE METADATA v3 — RUN SUMMARY")
print("=" * 65)
print(f"  run_timestamp   : {RUN_TS}")
print(f"  load_type       : {LOAD_TYPE}")
print(f"  partition       : {INGESTION_YEAR}/{INGESTION_MONTH}/{INGESTION_DAY or '*'}")
print(f"  bronze_rows     : {BRONZE_ROW_COUNT}")
print(f"  silver_rows     : {SILVER_ROW_COUNT}  (this run)")
print(f"  silver_total    : {silver_total}  (all-time)")
print(f"  quarantined     : {BRONZE_ROW_COUNT - SILVER_ROW_COUNT}")
print("=" * 65)

print("\nInvoices per month (Silver total):")
silver_df.groupBy("invoice_year", "invoice_month").count() \
         .orderBy("invoice_year", "invoice_month").show()